<table>
  <tr>
    <td><div align="left"><font size="30">Machine Vision Toolbox for Python — Interactive Demo</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/figs/VisionToolboxLogo_NoBackgnd@2x.png" width="400"></td>
  </tr>
</table>

This notebook runs entirely in your browser using [JupyterLite](https://jupyterlite.readthedocs.io) and [Pyodide](https://pyodide.org).
No installation required.  The cell below installs the toolbox the first time it is run (and will take a few seconds).

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install([
        "opencv-python",
        "spatialmath-python",
        "pgraph-python",
        "ansitable",
        "mvtb-data",
    ])

    import cv2  # noqa: F401 — force cv2 into module registry before toolbox import

    await micropip.install("machinevision-toolbox-python", deps=False)

print("Ready.")


## Images and pixels

The core class is `Image`.  Let's read one of the bundled images and inspect it.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from machinevisiontoolbox import Image

mona = Image.Read("monalisa.png")
print(f"size: {mona.width} x {mona.height}, dtype: {mona.dtype}, colour: {mona.iscolor}")


Display the image inline with `disp()`, which uses matplotlib.

In [ ]:
mona.disp()

<div class="alert alert-info">
<strong>Note:</strong> In this environment it is not possible to examine pixel values by hovering the mouse cursor over an image.  This a limitation of Matplotlib in a Pyodide Web Worker.
</div>

## Smoothing

Apply a Gaussian blur and display original and result side-by-side.

In [ ]:
smooth = mona.smooth(sigma=3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
mona.disp(ax=axes[0], title="Original")
smooth.disp(ax=axes[1], title="Smoothed (sigma=3)")
plt.tight_layout()

## Greyscale and histograms

In [ ]:
grey = mona.mono()
print(f"grey: {grey.width} x {grey.height}, planes: {grey.nplanes}")

hist, x = grey.hist()
plt.figure()
plt.bar(x, hist, width=1, color="steelblue")
plt.xlabel("Pixel value")
plt.ylabel("Count")
plt.title("Greyscale histogram")
plt.tight_layout()

## Colour planes

Access individual colour planes by name — the toolbox tracks the colour order so
you never need to worry about BGR vs RGB.

In [ ]:
flowers = Image.Read("flowers1.png")
print(f"colour order: {flowers.colororder_str}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, plane in zip(axes, ["R", "G", "B"]):
    flowers.plane(plane).disp(ax=ax, title=plane, colormap="gray")
plt.tight_layout()

## Binary blobs

Load a binary image and find the blobs.

In [ ]:
sharks = Image.Read("shark2.png")
blobs = sharks.blobs()
print(blobs)

fig, ax = plt.subplots()
sharks.disp(ax=ax)
blobs.plot_box(ax, color="g")
blobs.plot_centroid(ax, "o", color="y")
plt.tight_layout()

## Camera model

Create a central perspective camera and project a 3-D point into the image plane.

In [ ]:
from machinevisiontoolbox import CentralCamera
from spatialmath import SE3

cam = CentralCamera(f=0.015, rho=10e-6, imagesize=[1280, 1024],
                    pp=[640, 512], name="mycamera")
print(cam)

P = [0.3, 0.4, 3.0]
p = cam.project_point(P)
print(f"Projected pixel: {p}")

# Shift camera 100 mm to the right
p2 = cam.project_point(P, pose=SE3(0.1, 0, 0))
print(f"Projected pixel (camera shifted): {p2}")